/home/mojtaba.kananisarche


In [ ]:
%cd '/home/mojtaba.kananisarche/'
from typing import Optional
from Zh.src.utils.file_folder import create_folders
import os
import json
import pandas as pd
import dataframe_image as dfi

async def table_results(
results_directory_path : str,
output_directory_path  : Optional[str] = None,
output_file_path_prefix: Optional[str] = 'results',
cmap                   : Optional[str] = "Greys",
reverse_cmap           : Optional[bool] = False,
maximum_tries          : Optional[int] = 5,
)                      : 
    
    KEYS_TO_COLLECT       = ['index', 'test', 'val', 'train_report', 'val_report', 'test_report']
    output_directory_path = output_directory_path or results_directory_path
    create_folders(output_directory_path)
    output_file_name = None

    i = 0
    while output_file_name is None:
        i += 1
        if not os.path.exists(os.path.join(output_directory_path, f"{output_file_path_prefix}_{i}.png")):
            output_file_name = f"{output_file_path_prefix}_{i}.png"
    
    output_file_path = os.path.join(output_directory_path, output_file_name)

    data = []
    for file_name in os.listdir(results_directory_path):
        if file_name.endswith('.json'):
            file_path = os.path.join(results_directory_path, file_name)
            with open(file=file_path, mode='r') as json_file:
                j = json.load(json_file)
                try:
                    data.append(
                        {
                            k:v for k, v in j.items() if k in KEYS_TO_COLLECT
                        }
                    )
                except KeyError as e:
                    print("Unable")
                    continue


    data = pd.DataFrame(data = data).dropna(how='all')
    try:
        data.index = data['index'].astype(int).values
        data       = data.sort_index()
        data       = data.drop('index', axis=1)
    except:
        print("No Index")
        pass

    folded_by_column = list(data['test'][0].keys())[0]

    multi_index = pd.MultiIndex.from_tuples([
    (f'Split ({folded_by_column})', 'Test'),
    (f'Split ({folded_by_column})', 'Validation'),
    ('Report', 'Train'),
    ('Report', 'Validation'),
    ('Report', 'Test'),
    ])
    data.columns = multi_index

    data[(f'Split ({folded_by_column})', 'Test')] = data[(f'Split ({folded_by_column})', 'Test')].apply(lambda x: x[folded_by_column])
    data[(f'Split ({folded_by_column})', 'Validation')]  = data[(f'Split ({folded_by_column})', 'Validation')].apply(lambda x: x[folded_by_column])

    data[('MAE', 'Train')] = data[('Report', 'Train')].apply(lambda x: round(x['overall']['mae'], 3))
    data[('MAE', 'Validation')]   = data[('Report', 'Validation')].apply(lambda x: round(x['overall']['mae'], 3))
    data[('MAE', 'Test')]  = data[('Report', 'Test')].apply(lambda x: round(x['overall']['mae'], 3))

    data[('Accuracy', 'Train')] = data[('Report', 'Train')].apply(lambda x: round(x['overall']['accuracy'], 3))
    data[('Accuracy', 'Validation')]   = data[('Report', 'Validation')].apply(lambda x: round(x['overall']['accuracy'], 3))
    data[('Accuracy', 'Test')]  = data[('Report', 'Test')].apply(lambda x: round(x['overall']['accuracy'], 3))

    data = data.drop('Report', axis=1, level=0)
    data.columns.names = [None, 'Index']

    avg = data.mean(numeric_only=True)
    data.loc['Average'] = avg
    data = data.fillna('-')

    cmaps = [cmap, f"{cmap}_r"] if reverse_cmap else [f"{cmap}_r", cmap]

    data_styled = data.style.background_gradient(subset=[('Accuracy', 'Train'), 
                                                         ('Accuracy', 'Validation'), 
                                                         ('Accuracy', 'Test')], axis=None, vmin=0, vmax=1, cmap=cmaps[0]) \
                        .background_gradient(subset=[('MAE', 'Train'), 
                                                     ('MAE', 'Validation'), 
                                                     ('MAE', 'Test')], vmin=0, vmax=1, axis=None, cmap=cmaps[1]) \
                    .format(precision=3) \
                    .set_table_styles(
                        [{'selector': 'th', 'props': [('text-align', 'center')]},  # Center align column names
                            {'selector': 'td', 'props': [('text-align', 'center')]},  # Center align values
                            {'selector': 'thead th', 'props': [('text-align', 'center')]},  # Center align indices
                            {'selector': 'td:nth-child(2)', 'props': [('border-left', '4px solid black')]},  # Add line between index and Test
                            {'selector': 'td:nth-child(4)', 'props': [('border-left', '4px solid black')]},  # Add line between val and train_mae
                            {'selector': 'td:nth-child(7)', 'props': [('border-left', '4px solid black')]},  # Add line between test_mae and train_accuracy
                            {'selector': 'td:nth-child(2)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(3)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(4)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(5)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(6)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(7)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(8)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(9)', 'props': [('width', '5em')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(3)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(7)', 'props': [('border-left', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('border-top', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('border-bottom', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('font-weight', 'bold')]}
                            ]  # Add line above the last horizontal row
                    ) \
                    .set_properties(**{'text-align': 'center'})  # Ensure all text is center aligned

    for i in range(maximum_tries):
        try:
            await dfi.export_async(obj=data_styled, filename=output_file_path, table_conversion='playwright')
            print(f"Try Number {i} Succeeded.")
            return
        except:
            print(f"Try Number {i} Failed.")
    # return data_styled, output_file_path
    

await table_results(results_directory_path = '/home/mojtaba.kananisarche/Zh/outputs/2025-04-15|23:45:55|RNN_FACENET_5CV', reverse_cmap=False, cmap="coolwarm")


Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed\nCall log:\n  - taking element screenshot\n  -   - waiting for fonts to load...\n  -   - fonts loaded\n  -   - attempting scroll into view action\n  -     - waiting for element to be stable\n')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Call log:
  - taking element screenshot
  -   - waiting for fonts to load...
  -   - fonts loaded
  -   - attempting scroll into view action
  -     - waiting for element to be stable



Try Number 0 Failed.
Try Number 1 Failed.
Try Number 2 Succeeded.


In [315]:
await dfi.export_async(obj=data_styled, filename=output_file_path, table_conversion='playwright')

In [298]:
data2.columns.names = [None, 'Index']

In [299]:
data2

split (fold)            mae                      accuracy            \
Index           test  val     train       val      test     train       val   
0                [1]  [2]  0.177000  0.480000  0.649000  1.000000  0.883000   
1                [1]  [3]  0.137000  0.591000  0.462000  1.000000  0.812000   
2                [1]  [4]  0.116000  0.394000  0.648000  1.000000  0.870000   
4                [2]  [1]  0.204000  0.699000  0.577000  1.000000  0.863000   
5                [2]  [3]  0.151000  0.535000  0.501000  1.000000  0.844000   
6                [2]  [4]  0.253000  0.515000  0.611000  0.984000  0.894000   
7                [2]  [5]  0.195000  0.538000  0.600000  1.000000  0.869000   
8                [3]  [1]  0.185000  0.569000  0.487000  1.000000  0.863000   
9                [3]  [2]  0.112000  0.532000  0.471000  1.000000  0.805000   
10               [3]  [4]  0.215000  0.615000  0.649000  1.000000  0.852000   
11               [3]  [5]  0.212000  0.549000  0.515000  0.980000  0.799000   
12               [4]  [1]  0.165000  0.624000  0.690000  1.000000  0.887000   
13               [4]  [2]  0.104000  0.395000  0.556000  1.000000  0.887000   
14               [4]  [3]  0.153000  0.380000  0.486000  1.000000  0.927000   
15               [4]  [5]  0.167000  0.533000  0.553000  1.000000  0.869000   
16               [5]  [1]  0.182000  0.762000  0.436000  0.996000  0.797000   
17               [5]  [2]  0.174000  0.522000  0.514000  1.000000  0.917000   
18               [5]  [3]  0.245000  0.629000  0.549000  1.000000  0.830000   
19               [5]  [4]  0.318000  0.552000  0.750000  0.985000  0.880000   
Average            -    -  0.182368  0.548105  0.563368  0.997105  0.860421   

                   
Index        test  
0        0.793000  
1        0.844000  
2        0.766000  
4        0.853000  
5        0.827000  
6        0.865000  
7        0.868000  
8        0.899000  
9        0.806000  
10       0.892000  
11       0.812000  
12       0.833000  
13       0.778000  
14       0.861000  
15       0.866000  
16       0.905000  
17       0.883000  
18       0.841000  
19       0.862000  
Average  0.844947

In [188]:
data.mean(numeric_only=True)

mae       train    0.182368
          val      0.548105
          test     0.563368
accuracy  train    0.997105
          val      0.860421
          test     0.844947
dtype: float64

In [181]:
data[('report', 'train')].apply(lambda x: round(x['overall']['accuracy'], 3))

0     1.000
1     1.000
2     1.000
4     1.000
5     1.000
6     0.984
7     1.000
8     1.000
9     1.000
10    1.000
11    0.980
12    1.000
13    1.000
14    1.000
15    1.000
16    0.996
17    1.000
18    1.000
19    0.985
Name: (report, train), dtype: float64

In [156]:
multi_index = pd.MultiIndex.from_tuples([
('split', 'test'),
('split', 'val'),
('report', 'train'),
('report', 'val'),
('report', 'test'),
])

In [ ]:
data2[('split', 'test')] = data['test']
data2[('split', 'val')] = data['val']

In [147]:
data2

,"(split, test)","(split, val)"
0,{'fold': [1]},{'fold': [2]}
1,{'fold': [1]},{'fold': [3]}
2,{'fold': [1]},{'fold': [4]}
4,{'fold': [2]},{'fold': [1]}
5,{'fold': [2]},{'fold': [3]}
6,{'fold': [2]},{'fold': [4]}
7,{'fold': [2]},{'fold': [5]}
8,{'fold': [3]},{'fold': [1]}
9,{'fold': [3]},{'fold': [2]}
10,{'fold': [3]},{'fold': [4]}


In [ ]:
import os
import json
import pandas as pd
import dataframe_image as dfi

directory_path = '/home/mojtaba.kananisarche/Zh/outputs/2025-04-15|23:45:55|RNN_FACENET_5CV'
# directory_path = '/home/mojtaba.kananisarche/Zh/models/e7/RNNFacenet'

results = []
keys_to_collect = ['index', 'test', 'val', 'train_report', 'val_report', 'test_report']
for file_name in os.listdir(directory_path):
    if file_name.endswith('.json'):
        file_path = os.path.join(directory_path, file_name)
        with open(file=file_path, mode='r') as json_file:
            data = json.load(json_file)
            try:
                results.append(
                    {
                        k:v for k, v in data.items() if k in keys_to_collect
                    }
                )
            except KeyError as e:
                print("Unable")
                continue

p = [r for r in results if r != {}]
p = pd.DataFrame(p)
p.index = p['index'].astype(int).values
# p.index = p['index']
p['train_accuracy'] = p['train_report'].apply(lambda x: round(x['overall']['accuracy'], 3))
p['train_mae'] = p['train_report'].apply(lambda x: round(x['overall']['mae'], 3))
p['val_accuracy'] = p['val_report'].apply(lambda x: round(x['overall']['accuracy'], 3))
p['val_mae'] = p['val_report'].apply(lambda x: round(x['overall']['mae'], 3))
p['test_accuracy'] = p['test_report'].apply(lambda x: round(x['overall']['accuracy'], 3))
p['test_mae'] = p['test_report'].apply(lambda x: round(x['overall']['mae'], 3))
# Convert the 'test' column to a string to make it hashable
# p = p.sort_values(['test', 'val'])
p['test'] = p['test'].apply(lambda x: str(x))
p['val'] = p['val'].apply(lambda x: str(x))
# p = p.sort_values(['test', 'val']).reset_index(drop=True)
p = p.sort_index()

avg = round(p.iloc[:, 4:].mean(numeric_only=True), 3)
avg['val'] = '-'
avg['test'] = '-'
p.loc['Average'] = avg
p2 = p.drop(['train_report', 'val_report', 'test_report'], axis=1)
p2 = p2[['test', 'val', 'train_mae', 'val_mae', 'test_mae', 'train_accuracy', 'val_accuracy', 'test_accuracy']]
multi_index = pd.MultiIndex.from_tuples([
    ('split', 'test'),
    ('split', 'validation'),
    ('mae', 'train'),
    ('mae', 'validation'),
    ('mae', 'test'),
    ('accuracy', 'train'),
    ('accuracy', 'validation'),
    ('accuracy', 'test')
])
p2.columns = multi_index

# Adjust pandas display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Adjust the width to fit the content

cmap = "coolwarm"
image_path = os.path.join(directory_path, 'results.png')
df_styled = p2.style.background_gradient(subset=[('accuracy', 'train'), ('accuracy', 'validation'), ('accuracy', 'test')], axis=None, vmin=0, vmax=1, cmap=f'{cmap}_r') \
                    .background_gradient(subset=[('mae', 'train'), ('mae', 'validation'), ('mae', 'test')], vmin=0, vmax=2, axis=None, cmap=f'{cmap}') \
                   .format(precision=3) \
                   .set_table_styles(
                       [{'selector': 'th', 'props': [('text-align', 'center')]},  # Center align column names
                        {'selector': 'td', 'props': [('text-align', 'center')]},  # Center align values
                        {'selector': 'thead th', 'props': [('text-align', 'center')]},  # Center align indices
                        {'selector': 'td:nth-child(2)', 'props': [('border-left', '4px solid black')]},  # Add line between index and test
                        {'selector': 'td:nth-child(4)', 'props': [('border-left', '4px solid black')]},  # Add line between val and train_mae
                        {'selector': 'td:nth-child(7)', 'props': [('border-left', '4px solid black')]},  # Add line between test_mae and train_accuracy
                        {'selector': 'thead tr:nth-child(1) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                        {'selector': 'thead tr:nth-child(1) th:nth-child(3)', 'props': [('border-left', '4px solid black')]},
                        {'selector': 'thead tr:nth-child(1) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                        {'selector': 'thead tr:nth-child(2) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                        {'selector': 'thead tr:nth-child(2) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                        {'selector': 'thead tr:nth-child(2) th:nth-child(7)', 'props': [('border-left', '4px solid black')]},
                        {'selector': f'tr:nth-child({p.shape[0]})', 'props': [('border-top', '4px solid black')]}]  # Add line above the last horizontal row
                   ) \
                   .set_properties(**{'text-align': 'center'})  # Ensure all text is center aligned
# df_styled

In [115]:
df_styled

In [81]:
p2.sort_values([('split', 'test'), ('split', 'validation')])

split                                                  mae             \
         test                                    validation  train validation   
Average     -                                             -  0.179      0.492   
20          1          [4, 2, 9, 8, 10, 3, 12, 11, 6, 5, 7]  0.175      0.472   
24         10           [4, 2, 9, 8, 3, 12, 11, 6, 5, 1, 7]  0.162      0.497   
25         11           [4, 2, 9, 8, 10, 3, 12, 6, 5, 1, 7]  0.222      0.590   
26         12           [4, 2, 9, 8, 10, 3, 11, 6, 5, 1, 7]  0.213      0.655   
1          13  [20, 23, 21, 24, 22, 19, 16, 14, 15, 17, 18]  0.124      0.588   
2          14  [20, 23, 21, 24, 22, 13, 19, 16, 15, 17, 18]  0.169      0.546   
3          15  [20, 23, 21, 24, 22, 13, 19, 16, 14, 17, 18]  0.188      0.481   
4          16  [20, 23, 21, 24, 22, 13, 19, 14, 15, 17, 18]  0.163      0.464   
54         17  [20, 23, 21, 24, 22, 13, 19, 16, 14, 15, 18]  0.221      0.493   
5          18  [20, 23, 21, 24, 22, 13, 19, 16, 14, 15, 17]  0.138      0.564   
55         19  [20, 23, 21, 24, 22, 13, 16, 14, 15, 17, 18]  0.158      0.493   
50          2          [4, 9, 8, 10, 3, 12, 11, 6, 5, 1, 7]  0.163      0.477   
56         20  [23, 21, 24, 22, 13, 19, 16, 14, 15, 17, 18]  0.167      0.385   
27         21  [20, 23, 24, 22, 13, 19, 16, 14, 15, 17, 18]  0.148      0.438   
28         22  [20, 23, 21, 24, 13, 19, 16, 14, 15, 17, 18]  0.196      0.459   
57         23  [20, 21, 24, 22, 13, 19, 16, 14, 15, 17, 18]  0.166      0.456   
29         24  [20, 23, 21, 22, 13, 19, 16, 14, 15, 17, 18]  0.166      0.492   
6          25  [28, 29, 36, 31, 30, 27, 32, 26, 33, 34, 35]  0.178      0.510   
7          26  [28, 29, 36, 31, 30, 25, 27, 32, 33, 34, 35]  0.197      0.501   
58         27  [28, 29, 36, 31, 30, 25, 32, 26, 33, 34, 35]  0.220      0.472   
30         28  [29, 36, 31, 30, 25, 27, 32, 26, 33, 34, 35]  0.152      0.469   
59         29  [28, 36, 31, 30, 25, 27, 32, 26, 33, 34, 35]  0.182      0.578   
21          3          [4, 2, 9, 8, 10, 12, 11, 6, 5, 1, 7]  0.206      0.493   
41         30  [28, 29, 36, 31, 25, 27, 32, 26, 33, 34, 35]  0.235      0.449   
31         31  [28, 29, 36, 30, 25, 27, 32, 26, 33, 34, 35]  0.169      0.541   
8          32  [28, 29, 36, 31, 30, 25, 27, 26, 33, 34, 35]  0.154      0.463   
9          33  [28, 29, 36, 31, 30, 25, 27, 32, 26, 34, 35]  0.187      0.415   
10         34  [28, 29, 36, 31, 30, 25, 27, 32, 26, 33, 35]  0.234      0.558   
11         35  [28, 29, 36, 31, 30, 25, 27, 32, 26, 33, 34]  0.177      0.607   
32         36  [28, 29, 31, 30, 25, 27, 32, 26, 33, 34, 35]  0.204      0.421   
12         37  [38, 41, 40, 42, 43, 39, 44, 48, 45, 47, 46]  0.160      0.516   
42         38  [41, 40, 42, 43, 39, 44, 48, 45, 47, 46, 37]  0.148      0.412   
33         39  [38, 41, 40, 42, 43, 44, 48, 45, 47, 46, 37]  0.156      0.471   
51          4          [2, 9, 8, 10, 3, 12, 11, 6, 5, 1, 7]  0.180      0.528   
34         40  [38, 41, 42, 43, 39, 44, 48, 45, 47, 46, 37]  0.189      0.542   
35         41  [38, 40, 42, 43, 39, 44, 48, 45, 47, 46, 37]  0.153      0.341   
36         42  [38, 41, 40, 43, 39, 44, 48, 45, 47, 46, 37]  0.261      0.497   
43         43  [38, 41, 40, 42, 39, 44, 48, 45, 47, 46, 37]  0.148      0.450   
44         44  [38, 41, 40, 42, 43, 39, 48, 45, 47, 46, 37]  0.137      0.599   
13         45  [38, 41, 40, 42, 43, 39, 44, 48, 47, 46, 37]  0.145      0.485   
14         46  [38, 41, 40, 42, 43, 39, 44, 48, 45, 47, 37]  0.170      0.410   
15         47  [38, 41, 40, 42, 43, 39, 44, 48, 45, 46, 37]  0.143      0.437   
16         48  [38, 41, 40, 42, 43, 39, 44, 45, 47, 46, 37]  0.154      0.347   
45         49  [55, 56, 53, 57, 58, 59, 50, 51, 54, 60, 52]  0.179      0.395   
52          5          [4, 2, 9, 8, 10, 3, 12, 11, 6, 1, 7]  0.123      0.479   
46         50  [55, 56, 53, 49, 57, 58, 59, 51, 54, 60, 52]  0.176      0.415   
37         51  [55, 56, 53, 49, 57, 58,

In [8]:
import pandas as pd

# Your list of column names
columns = ['test', 'val', 'train_mae', 'val_mae', 'test_mae', 'train_accuracy', 'val_accuracy', 'test_accuracy']

# Create a sample DataFrame (replace with your actual data)
data = {col: [i for i in range(5)] for col in columns}
df = pd.DataFrame(data)

# Create the multi-index
multi_index = pd.MultiIndex.from_tuples([
    ('split', 'test'),
    ('split', 'val'),
    ('mae', 'train'),
    ('mae', 'validation'),
    ('mae', 'test'),
    ('accuracy', 'train'),
    ('accuracy', 'validation'),
    ('accuracy', 'test')
])

# Assign the multi-index to the DataFrame's columns
df.columns = multi_index

# Rename the columns at the top level
df = df.rename(columns={'test': 'test', 'val': 'val'}, level=0)

In [13]:
p2.columns = multi_index

In [14]:
p2

split                   mae                   accuracy  \
                  test            val  train validation   test    train   
0        {'fold': [1]}  {'fold': [2]}  0.177      0.480  0.649    1.000   
1        {'fold': [1]}  {'fold': [3]}  0.137      0.591  0.462    1.000   
2        {'fold': [1]}  {'fold': [4]}  0.116      0.394  0.648    1.000   
3        {'fold': [2]}  {'fold': [1]}  0.204      0.699  0.577    1.000   
4        {'fold': [2]}  {'fold': [3]}  0.151      0.535  0.501    1.000   
5        {'fold': [2]}  {'fold': [4]}  0.253      0.515  0.611    0.984   
6        {'fold': [2]}  {'fold': [5]}  0.195      0.538  0.600    1.000   
7        {'fold': [3]}  {'fold': [1]}  0.185      0.569  0.487    1.000   
8        {'fold': [3]}  {'fold': [2]}  0.112      0.532  0.471    1.000   
9        {'fold': [3]}  {'fold': [4]}  0.215      0.615  0.649    1.000   
10       {'fold': [3]}  {'fold': [5]}  0.212      0.549  0.515    0.980   
11       {'fold': [4]}  {'fold': [1]}  0.165      0.624  0.690    1.000   
12       {'fold': [4]}  {'fold': [2]}  0.104      0.395  0.556    1.000   
13       {'fold': [4]}  {'fold': [3]}  0.153      0.380  0.486    1.000   
14       {'fold': [4]}  {'fold': [5]}  0.167      0.533  0.553    1.000   
15       {'fold': [5]}  {'fold': [1]}  0.182      0.762  0.436    0.996   
16       {'fold': [5]}  {'fold': [2]}  0.174      0.522  0.514    1.000   
17       {'fold': [5]}  {'fold': [3]}  0.245      0.629  0.549    1.000   
18       {'fold': [5]}  {'fold': [4]}  0.318      0.552  0.750    0.985   
Average              -              -  0.182      0.548  0.563    0.997   

                           
        validation   test  
0            0.883  0.793  
1            0.812  0.844  
2            0.870  0.766  
3            0.863  0.853  
4            0.844  0.827  
5            0.894  0.865  
6            0.869  0.868  
7            0.863  0.899  
8            0.805  0.806  
9            0.852  0.892  
10           0.799  0.812  
11           0.887  0.833  
12           0.887  0.778  
13           0.927  0.861  
14           0.869  0.866  
15           0.797  0.905  
16           0.917  0.883  
17           0.830  0.841  
18           0.880  0.862  
Average      0.860  0.845

In [ ]:
cmap = "coolwarm"
image_path = os.path.join(directory_path, 'results.png')
df_styled = p2.style.background_gradient(subset=['train_accuracy', 'val_accuracy', 'test_accuracy'], axis=None, vmin=0, vmax=1, cmap=f'{cmap}_r') \
                    .background_gradient(subset=['train_mae', 'val_mae', 'test_mae'], vmin=0, vmax=2, axis=None, cmap=f'{cmap}') \
                   .format(precision=3) \
                   .set_table_styles(
                       [{'selector': 'th', 'props': [('text-align', 'center')]},  # Center align column names
                        {'selector': 'td', 'props': [('text-align', 'center')]},  # Center align values
                        {'selector': 'thead th', 'props': [('text-align', 'center')]}]  # Center align indices
                   ) \
                   .set_properties(**{'text-align': 'center'})  # Ensure all text is center aligned

dfi.export(df_styled, image_path, table_conversion='matplotlib')

In [ ]:
"Accent"
"Accent_r"
"Blues"
"Blues_r"
"BrBG"
"BrBG_r"
"BuGn"
"BuGn_r"
"BuPu"
"BuPu_r"
"CMRmap"
"CMRmap_r"
"Dark2"
"Dark2_r"
"GnBu"
"GnBu_r"
"Grays"
"Greens"
"Greens_r"
"Greys"
"Greys_r"
"OrRd"
"OrRd_r"
"Oranges"
"Oranges_r"
"PRGn"
"PRGn_r"
"Paired"
"Paired_r"
"Pastel1"
"Pastel1_r"
"Pastel2"
"Pastel2_r"
"PiYG"
"PiYG_r"
"PuBu"
"PuBuGn"
"PuBuGn_r"
"PuBu_r"
"PuOr"
"PuOr_r"
"PuRd"
"PuRd_r"
"Purples"
"Purples_r"
"RdBu"
"RdBu_r"
"RdGy"
"RdGy_r"
"RdPu"
"RdPu_r"
"RdYlBu"
"RdYlBu_r"
"RdYlGn"
"RdYlGn_r"
"Reds"
"Reds_r"
"Set1"
"Set1_r"
"Set2"
"Set2_r"
"Set3"
"Set3_r"
"Spectral"
"Spectral_r"
"Wistia"
"Wistia_r"
"YlGn"
"YlGnBu"
"YlGnBu_r"
"YlGn_r"
"YlOrBr"
"YlOrBr_r"
"YlOrRd"
"YlOrRd_r"
"afmhot"
"afmhot_r"
"autumn"
"autumn_r"
"binary"
"binary_r"
"bone"
"bone_r"
"brg"
"brg_r"
"bwr"
"bwr_r"
"cividis"
"cividis_r"
"cool"
"cool_r"
"coolwarm"
"coolwarm_r"
"copper"
"copper_r"
"cubehelix"
"cubehelix_r"
"flag"
"flag_r"
"gist_earth"
"gist_earth_r"
"gist_gray"
"gist_gray_r"
"gist_grey"
"gist_heat"
"gist_heat_r"
"gist_ncar"
"gist_ncar_r"
"gist_rainbow"
"gist_rainbow_r"
"gist_stern"
"gist_stern_r"
"gist_yarg"
"gist_yarg_r"
"gist_yerg"
"gnuplot"
"gnuplot2"
"gnuplot2_r"
"gnuplot_r"
"gray"
"gray_r"
"grey"
"hot"
"hot_r"
"hsv"
"hsv_r"
"inferno"
"inferno_r"
"jet"
"jet_r"
"magma"
"magma_r"
"nipy_spectral"
"nipy_spectral_r"
"ocean"
"ocean_r"
"pink"
"pink_r"
"plasma"
"plasma_r"
"prism"
"prism_r"
"rainbow"
"rainbow_r"
"seismic"
"seismic_r"
"spring"
"spring_r"
"summer"
"summer_r"
"tab10"
"tab10_r"
"tab20"
"tab20_r"
"tab20b"
"tab20b_r"
"tab20c"
"tab20c_r"
"terrain"
"terrain_r"
"turbo"
"turbo_r"
"twilight"
"twilight_r"
"twilight_shifted"
"twilight_shifted_r"
"viridis"
"viridis_r"
"winter"
"winter_r"

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

ModuleNotFoundError: No module named 'pyarrow'

In [ ]:
pd.read_parquet(path = "/work/long_lab/Mojtaba/data/UTA/1.3.3-preprocessed/facenet_preprocessed_alternativemapping.parquet", row)

In [ ]:
import os
file_path = "/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes"
depth = 1
# for os.walk(file_path)

In [ ]:
import os
file_path = "/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes"
depth = 2
EXT_LIST = []
file_name = []
for dirpath, dirnames, file_names in os.walk(file_path):
    print(dirpath, len(file_path))
    if dirpath[len(file_path):].count(os.sep) < depth:
        file_name.extend([os.path.join(dirpath, file_name) for file_name in file_names if not EXT_LIST or os.path.splitext(file_name)[1].lower() in EXT_LIST])

/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes 56
/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes/.ipynb_checkpoints 56
/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes/Summary 56
/home/mojtaba.kananisarche/Zh/data/UTA/1.1.2-blendshapes/Summary/.ipynb_checkpoints 56


In [ ]:
EXT_LIST = ['ASv']
[ext.lower() if ext.startswith('.') else '.'+ext.lower() for ext in EXT_LIST]

['.asv']

In [ ]:
import cv2
import os

# Path to the video file
video_path = "/work/long_lab/Mojtaba/augmented/brightness_alpha0.8_beta-10/1_06_5_30fps_brightness_alpha0.8_beta-10.mp4"

# Output directory
output_path = "/home/mojtaba.kananisarche/Zh/temp/out2"
os.makedirs(output_path, exist_ok=True)

# Open the video file
cap = cv2.VideoCapture(video_path)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / fps
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Save the first 2 seconds of the video
output_file = os.path.join(output_path, "first_2_seconds.mp4")
out = cv2.VideoWriter(output_file, fourcc, fps, (width, height))

frames_to_save = int(fps * 2)  # Number of frames for 2 seconds
frame_counter = 0

while cap.isOpened() and frame_counter < frames_to_save:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)
    frame_counter += 1

# Release resources
cap.release()
out.release()

print(f"First 2 seconds of the video saved at: {output_file}")

First 2 seconds of the video saved at: /home/mojtaba.kananisarche/Zh/temp/out2/first_2_seconds.mp4


In [ ]:
import cv2

def get_video_properties(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Unable to open video file: {video_path}")
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps if fps > 0 else 0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    cap.release()
    return {
        "fps": fps,
        "duration": duration,
        "width": width,
        "height": height
    }

def compare_videos(video1_path, video2_path):
    video1_props = get_video_properties(video1_path)
    video2_props = get_video_properties(video2_path)
    
    differences = {}
    for key in video1_props:
        if video1_props[key] != video2_props[key]:
            differences[key] = (video1_props[key], video2_props[key])
    
    if differences:
        print("Differences found between the two videos:")
        for prop, values in differences.items():
            print(f"{prop}: Video 1 = {values[0]}, Video 2 = {values[1]}")
    else:
        print("The two videos are identical in terms of FPS, duration, width, and height.")

# Paths to the two video files
video1_path = "/home/mojtaba.kananisarche/Zh/temp/out2/first_2_seconds.mp4"
video2_path = "/home/mojtaba.kananisarche/Zh/temp/out2/video/brightness_alpha0.8_beta-10/1_06_5_30fps_brightness_alpha0.8_beta-10.mp4"

# Compare the videos
compare_videos(video1_path, video2_path)

The two videos are identical in terms of FPS, duration, width, and height.


In [ ]:
import ffm

In [ ]:
import multiprocessing
import time
from functools import partial

class ParallelProcessor:
    def __init__(self, data, multiplier):
        self.data = data
        self.multiplier = multiplier  # Property that will be used in the method

    def process_item(self, x):
        time.sleep(1)  # Simulate a time-consuming operation
        return (x * self.multiplier)  # Use both the input and the class property

    def process_data(self):
        # Use functools.partial to bind the instance method and class property
        func = partial(self.process_item)
        
        # Create a pool of worker processes
        with multiprocessing.Pool(4) as pool:
            # Map the function to the data in parallel
            results = pool.map(func, self.data)
        
        return results

if __name__ == "__main__":
    data = [1, 2, 3, 4, 5]
    multiplier = 2  # Class property that will be used
    processor = ParallelProcessor(data, multiplier)
    results = processor.process_data()
    print(f"Results: {results}")


Results: [2, 4, 6, 8, 10]


In [ ]:
%cd '/home/mojtaba.kananisarche/'

/home/mojtaba.kananisarche


In [ ]:
from Zh.src.utils.video import *
import cv2

In [ ]:
path = '/work/long_lab/Mojtaba/augmented/cropped/1_08_0_30fps_cropped.mp4'
cap = cv2.VideoCapture(path)
get_video_info(cap)

(30, 174, 174, 18158)

: 

In [ ]:
from Zh.src.utils.face_detection.face_detector import *

2025-04-02 04:24:05.194800: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-02 04:24:05.203115: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-02 04:24:05.233913: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743589445.287291 2442567 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743589445.303647 2442567 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743589445.338599 2442567 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
!pwd

/home/mojtaba.kananisarche/Zh/temp


In [ ]:
from Zh.src.utils.face_detection.face_detector import *
from Zh.src.utils.crop.cropper import *
from Zh.src.utils.video import *
import cv2
import os
import mediapipe as mp

In [ ]:
detector = FaceDetector(False, False, 1).landmarker

I0000 00:00:1743584268.745807 2183098 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1743584268.755812 2194120 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.1.4), renderer: llvmpipe (LLVM 17.0.6, 256 bits)
W0000 00:00:1743584268.758815 2183098 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1743584268.780514 2194126 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743584268.820243 2194139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
input_folder='/home/mojtaba.kananisarche/Zh/data/UTA/1-raw_30fps'
video_file = os.listdir(input_folder)[1]
file_path = os.path.join(input_folder, video_file)
print(file_path)

/home/mojtaba.kananisarche/Zh/data/UTA/1-raw_30fps/1_04_0_30fps.mp4


In [ ]:
cap = cv2.VideoCapture(file_path)
fps, width, height, total_frames = get_video_info(cap)
for i in range(10):
    ret, frame = cap.read()
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    d = detector.detect(mp_image)
    cropped_face = crop_face_from_frame(frame, d, width, height, margin=0.1)
    print(cropped_face)
    

[[[ 91  93  97]
  [ 91  93  97]
  [ 91  93  97]
  ...
  [ 66  57  55]
  [ 66  57  55]
  [ 65  56  54]]

 [[ 91  93  97]
  [ 91  93  97]
  [ 91  93  97]
  ...
  [ 66  57  55]
  [ 66  57  55]
  [ 64  55  53]]

 [[ 91  93  97]
  [ 91  93  97]
  [ 91  93  97]
  ...
  [ 66  57  55]
  [ 65  56  54]
  [ 64  55  53]]

 ...

 [[253 253 251]
  [254 254 252]
  [254 254 252]
  ...
  [224 224 222]
  [222 222 219]
  [219 220 216]]

 [[253 253 251]
  [254 254 252]
  [254 254 252]
  ...
  [222 222 220]
  [221 221 219]
  [219 220 216]]

 [[253 253 251]
  [254 254 252]
  [254 254 252]
  ...
  [222 222 220]
  [221 221 219]
  [219 219 217]]]
[[[ 91  93  97]
  [ 91  93  97]
  [ 91  93  97]
  ...
  [ 66  57  55]
  [ 66  57  55]
  [ 64  55  53]]

 [[ 91  93  97]
  [ 91  93  97]
  [ 91  93  97]
  ...
  [ 66  57  55]
  [ 66  57  55]
  [ 64  55  53]]

 [[ 92  94  98]
  [ 92  94  98]
  [ 92  94  98]
  ...
  [ 65  56  54]
  [ 64  55  53]
  [ 64  55  53]]

 ...

 [[254 254 252]
  [254 254 252]
  [254 254 252]
  ..

In [ ]:
from Zh.src.process.face_video_processor import *

2025-04-02 03:51:22.385566: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-02 03:51:22.393411: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-02 03:51:22.423780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743587482.476808 2335868 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743587482.493079 2335868 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743587482.527123 2335868 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
input_folder='/home/mojtaba.kananisarche/Zh/data/UTA/1-raw_30fps'
output_folder='/home/mojtaba.kananisarche/Zh/temp/out'

vp = VideoProcessor(input_folder=input_folder,
                    output_folder=output_folder,
                    use_tqdm=True)

I0000 00:00:1743587489.750722 2335868 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1743587489.780285 2339379 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.1.4), renderer: llvmpipe (LLVM 17.0.6, 256 bits)
W0000 00:00:1743587489.781225 2335868 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1743587489.823174 2339383 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743587489.865629 2339398 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
augmentations = {
    "gaussianblur": [
        {"kernel_size": 3},
        {"kernel_size": 5}
    ],
    "brightness": [
        {"alpha": 1.2, "beta": 10},
        {"alpha": 0.8, "beta": -10}
    ],
    "contrast": [
        {"alpha": 1.5, "beta": 0},
        {"alpha": 0.7, "beta": 0}
    ],
    "flip": [
        {"flip_code": 1},  # Horizontal flip
        {"flip_code": 0},  # Vertical flip
        {"flip_code": -1}  # Both horizontal and vertical
    ],
    "grayscale": [{}],  # No parameters needed
    "rotate": [
        {"angle": 15},
        {"angle": -15}
    ]
}
    
vp.process_single_video(
    os.listdir(input_folder)[1],
    0,
    1,
    augmentations,
    True,
    True
)

Processing 1_04_0_30fps.mp4:   0%|          | 0/30 [00:00<?, ?frame/s]

frame = 0.0


Processing 1_04_0_30fps.mp4:  10%|█         | 3/30 [00:00<00:03,  7.15frame/s]

frame = 1.0
frame = 2.0
frame = 3.0


Processing 1_04_0_30fps.mp4:  17%|█▋        | 5/30 [00:00<00:02,  9.61frame/s]

...

frame = 25.0
frame = 26.0
frame = 27.0
frame = 28.0
frame = 29.0
saving


Saving 1_04_0_30fps_cropped.mp4: 100%|██████████| 30/30 [00:00<00:00, 998.20frame/s]


saved /home/mojtaba.kananisarche/Zh/temp/out/cropped/1_04_0_30fps_cropped.mp4


Saving 1_04_0_30fps_gaussian_blur_kernelsize-3.mp4: 100%|██████████| 30/30 [00:00<00:00, 1191.99frame/s]
Saving 1_04_0_30fps_gaussian_blur_kernelsize-5.mp4: 100%|██████████| 30/30 [00:00<00:00, 1210.79frame/s]
Saving 1_04_0_30fps_brightness_alpha-1.2_beta-10.mp4: 100%|██████████| 30/30 [00:00<00:00, 1186.49frame/s]
Saving 1_04_0_30fps_brightness_alpha-0.8_beta--10.mp4: 100%|██████████| 30/30 [00:00<00:00, 1214.39frame/s]
Saving 1_04_0_30fps_contrast_alpha-1.5_beta-0.mp4: 100%|██████████| 30/30 [00:00<00:00, 1115.07frame/s]
Saving 1_04_0_30fps_contrast_alpha-0.7_beta-0.mp4: 100%|██████████| 30/30 [00:00<00:00, 1228.19frame/s]
Saving 1_04_0_30fps_rotate_angle--15.mp4: 100%|██████████| 30/30 [00:00<00:00, 1210.54frame/s]


{'cropped': '/home/mojtaba.kananisarche/Zh/temp/out/cropped/1_04_0_30fps_cropped.mp4',
 'gaussian_blur_kernelsize-3': '/home/mojtaba.kananisarche/Zh/temp/out/gaussian_blur_kernelsize-3/1_04_0_30fps_gaussian_blur_kernelsize-3.mp4',
 'gaussian_blur_kernelsize-5': '/home/mojtaba.kananisarche/Zh/temp/out/gaussian_blur_kernelsize-5/1_04_0_30fps_gaussian_blur_kernelsize-5.mp4',
 'brightness_alpha-1.2_beta-10': '/home/mojtaba.kananisarche/Zh/temp/out/brightness_alpha-1.2_beta-10/1_04_0_30fps_brightness_alpha-1.2_beta-10.mp4',
 'brightness_alpha-0.8_beta--10': '/home/mojtaba.kananisarche/Zh/temp/out/brightness_alpha-0.8_beta--10/1_04_0_30fps_brightness_alpha-0.8_beta--10.mp4',
 'contrast_alpha-1.5_beta-0': '/home/mojtaba.kananisarche/Zh/temp/out/contrast_alpha-1.5_beta-0/1_04_0_30fps_contrast_alpha-1.5_beta-0.mp4',
 'contrast_alpha-0.7_beta-0': '/home/mojtaba.kananisarche/Zh/temp/out/contrast_alpha-0.7_beta-0/1_04_0_30fps_contrast_alpha-0.7_beta-0.mp4',
 'flip_flipcode-1': '/home/mojtaba.kanan

In [ ]:
import subprocess
import sys

# # List of required packages
# required_packages = [
#     'opencv-python',
#     'mediapipe',
#     'pandas',
#     'numpy',
#     'tqdm',
#     'scikit-learn',
#     'mord',
#     'tensorflow',
#     'matplotlib',
#     'tensorflow-addons',
#     'imbalanced-learn'
# ]

# # Function to install packages if not already installed
# def install_package(package):
#     subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# # Check and install missing packages
# for package in required_packages:
#     try:
#         __import__(package)
#     except ImportError:
#         print(f"{package} not found, installing...")
#         install_package(package)

# Now, import the libraries

import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
import logging
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mord import LogisticAT  # Ordinal logistic regression
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, root_mean_squared_error
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
from keras import backend as K
from keras import losses
from keras.layers import Lambda
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from keras.preprocessing.sequence import pad_sequences
import random
import concurrent.futures
from tqdm import tqdm
from datetime import datetime
import json

2025-03-10 00:59:30.544042: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-10 00:59:33.066353: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-10 00:59:40.642650: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-10 00:59:40.642721: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-10 00:59:42.702394: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("process_video.log"),
        logging.StreamHandler()
    ]
)


In [ ]:
# Function to set the random seed
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.config.experimental.enable_op_determinism()


In [ ]:
def replace_target(value, mapping):
    return mapping[str(value)]

In [ ]:
def step(val, steps, outputs):
    for i in range(len(steps)-1):
        low = steps[i]
        high = steps[i+1]
        if val >= low and val <= high:
            return outputs[i]
        

In [ ]:
def get_prediction(raw_pred):
    # Clip to ensure within range
    clipped = np.clip(raw_pred, 1, 9)
    # Round to nearest integer
    return np.round(clipped).astype(int)


In [ ]:
def reshape_data_rnn(df, feature_columns=[], max_slices=30, sortby_column='slice_index', target_column='target'):
    X, y = [], []
    
    grouped = df.groupby('filename')
    
    for video, group in grouped:
        # Sort by slice_index to maintain temporal order
        group = group.sort_values(by=sortby_column)

        # Extract features
        slices = group[feature_columns].values.tolist()

        # Split into chunks of max_slices
        for i in range(0, len(slices), max_slices):
            chunk = slices[i:i + max_slices]
            X.append(chunk)
            y.append(group[target_column].iloc[0])  # Same label for all chunks from the same video

    # Pad sequences to the max_slices length
    X_padded = pad_sequences(X, maxlen=max_slices, padding='post', dtype='float32')

    # Convert labels to numpy array
    y = np.array(y)
    
    return X_padded, y



In [ ]:
def ordinal_loss(y_true, y_pred):
    # Round the prediction to get integer values
    y_pred_rounded = tf.round(y_pred)
    # Clip to ensure values are between 1 and 9
    y_pred_clipped = tf.clip_by_value(y_pred_rounded, 1, 9)
    # Calculate MSE with the clipped values
    return tf.reduce_mean(tf.square(y_true - y_pred_clipped))


In [ ]:
logging.info(f"Starting")
set_seed(42)

2025-03-10 01:02:44,262 - INFO - Starting


In [ ]:
dataset_name = "Blendshapes_Summary_100_100_0.6_45_3.csv"
df = pd.read_csv(f"/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/{dataset_name}", index_col=0)
print(df.shape)

df.head()

(32683, 282)


,slice_index,filename,start_frame,end_frame,_neutral_mean,browDownLeft_mean,browDownRight_mean,browInnerUp_mean,browOuterUpLeft_mean,browOuterUpRight_mean,...,blinks_median_duration,blinks_std_duration,blinks_min_duration,blinks_max_duration,naps_count,naps_mean_duration,naps_median_duration,naps_std_duration,naps_min_duration,naps_max_duration
0,0,2_21_10_30fps_blendshape.csv,0,99,0.0,0.0083,0.0047,0.1579,0.2362,0.2081,...,3.0,2.309401,3,7,0,0.0,0.0,0.0,0,0
1,1,2_21_10_30fps_blendshape.csv,100,199,0.0,0.0074,0.0031,0.1580,0.2405,0.2433,...,4.0,0.000000,4,4,0,0.0,0.0,0.0,0,0
2,2,2_21_10_30fps_blendshape.csv,200,299,0.0,0.0117,0.0083,0.1100,0.2051,0.1928,...,6.5,3.535534,4,9,0,0.0,0.0,0.0,0,0
3,3,2_21_10_30fps_blendshape.csv,300,399,0.0,0.0194,0.0098,0.0821,0.1609,0.1653,...,5.0,0.000000,5,5,0,0.0,0.0,0.0,0,0
4,4,2_21_10_30fps_blendshape.csv,400,499,0.0,0.0170,0.0092,0.0900,0.1756,0.1827,...,3.0,1.000000,2,4,0,0.0,0.0,0.0,0,0


In [ ]:
mapping = {
    '10' : 8.5,
    '5' : 6.5,
    '0' : 2
}

df['fold'] = df['filename'].apply(lambda x: int(x.split('_')[0]))
df['subject_id'] = df['filename'].apply(lambda x: int(x.split('_')[1]))
df['target'] = df['filename'].apply(lambda x:mapping.get(x.split('_')[2]))
# df.drop('filename', axis=1, inplace=True)

df.head()

,slice_index,filename,start_frame,end_frame,_neutral_mean,browDownLeft_mean,browDownRight_mean,browInnerUp_mean,browOuterUpLeft_mean,browOuterUpRight_mean,...,blinks_max_duration,naps_count,naps_mean_duration,naps_median_duration,naps_std_duration,naps_min_duration,naps_max_duration,fold,subject_id,target
0,0,2_21_10_30fps_blendshape.csv,0,99,0.0,0.0083,0.0047,0.1579,0.2362,0.2081,...,7,0,0.0,0.0,0.0,0,0,2,21,8.5
1,1,2_21_10_30fps_blendshape.csv,100,199,0.0,0.0074,0.0031,0.1580,0.2405,0.2433,...,4,0,0.0,0.0,0.0,0,0,2,21,8.5
2,2,2_21_10_30fps_blendshape.csv,200,299,0.0,0.0117,0.0083,0.1100,0.2051,0.1928,...,9,0,0.0,0.0,0.0,0,0,2,21,8.5
3,3,2_21_10_30fps_blendshape.csv,300,399,0.0,0.0194,0.0098,0.0821,0.1609,0.1653,...,5,0,0.0,0.0,0.0,0,0,2,21,8.5
4,4,2_21_10_30fps_blendshape.csv,400,499,0.0,0.0170,0.0092,0.0900,0.1756,0.1827,...,4,0,0.0,0.0,0.0,0,0,2,21,8.5


In [ ]:
def report(model, x, y_true, unique_classes):
    y_true = np.array(y_true)
    y_pred = np.array(model.predict(x, verbose=0)).flatten()
    results = {}
    for cls in unique_classes:
        idx = y_true == cls
        if np.all(idx == False):
            results[cls] = -1
            continue
        mae = root_mean_squared_error(y_true[idx], y_pred[idx])
        results[cls] = mae

    return y_pred, results

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer)):
            return int(obj)
        elif isinstance(obj, (np.floating)):
            return round(float(obj), 1)
        elif isinstance(obj, (np.ndarray)):
            return [round(float(x), 1) if isinstance(x, (np.floating, float)) else x for x in obj.tolist()]
        return super(NumpyEncoder, self).default(obj)

In [ ]:

class Model:
    def __init__(self, units, dropout_ratio, kernel_regularizer_l2, kernel_regularizer_l1, batch_size, num_classes, learning_rate, weight_decay, beta_1, beta_2, epsilon, epochs, patience, max_slices, dataset_name):
        self.units = units
        self.dropout_ratio = dropout_ratio
        self.kernel_regularizer_l2 = kernel_regularizer_l2
        self.kernel_regularizer_l1 = kernel_regularizer_l1
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.epochs = epochs
        self.patience = patience
        self.max_slices = max_slices
        self.dataset_name = dataset_name
        self.dataset_id = self.dataset_name.split("Blendshapes_Summary_")[1].split(".csv")[0]
        self.models = []
        self.histories = []
        self.reports = []
        
        
    def build_model(self, input_shape):
        model = keras.Sequential()
        model.add(layers.Input(shape=(input_shape[0], input_shape[1])))

        # Add LSTM layers
        for u in self.units[:-1]:
            model.add(layers.LSTM(u, activation='tanh', return_sequences=True, 
                                  kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
            model.add(layers.BatchNormalization())
            model.add(layers.Dropout(self.dropout_ratio))

        # Final LSTM layer without return_sequences
        model.add(layers.LSTM(self.units[-1], activation='tanh', 
                              kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(self.dropout_ratio))

        # Final output layer
        model.add(layers.Dense(1, activation='sigmoid'))  # Use sigmoid activation

        # Use a custom output layer to scale the output to 1-9
        model.add(layers.Lambda(lambda x: x * 8 + 1)) 

        model.compile(
            optimizer = keras.optimizers.AdamW(
                learning_rate=self.learning_rate,
                weight_decay=self.weight_decay,
                beta_1=self.beta_1,
                beta_2=self.beta_2,
                epsilon=self.epsilon
            ),
            loss='mse',
            metrics=['mae']
        )

        return model



    # First, define the train_single_fold function outside of the class
    def _train_single_fold(self, args):
        """Helper function for parallel processing in test_model.

        Args:
            args: Tuple containing (model_instance, test_idx, val_idx, data)

        Returns:
            Dictionary containing model, history, and reports
        """
        self, test_subject_idx, data = args
        local_results = {}

        subject_fold = data[data['subject_id']==test_subject_idx]['fold'].values[0]
        validation_subjects_idx = data[(data['subject_id']!=test_subject_idx) & (data['fold']==subject_fold)]['subject_id'].unique().tolist()
        logging.info(f"Starting Test {test_subject_idx} & Validation {validation_subjects_idx}")

        train = data[~data['subject_id'].isin(validation_subjects_idx + [test_subject_idx])]
        val   = data[data['subject_id'].isin(validation_subjects_idx)]
        test  = data[data['subject_id']==test_subject_idx]
        logging.info(f"train {train.shape} & val {val.shape} & test {test.shape}")

        # List of columns to exclude from scaling
        exclude_cols = ['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id']
        feature_columns = data.columns.difference(exclude_cols)

        # Fit the scaler on the training data
        scaler = StandardScaler().fit(train[feature_columns])

        # Transform the training, validation, and test data
        train_scaled = scaler.transform(train[feature_columns])
        val_scaled   = scaler.transform(val[feature_columns]  )
        test_scaled  = scaler.transform(test[feature_columns] )

        # Convert scaled arrays back to DataFrames and add the excluded columns
        train_scaled_df = pd.DataFrame(train_scaled, columns=feature_columns, index=train.index)
        val_scaled_df   = pd.DataFrame(val_scaled,   columns=feature_columns, index=val.index  )
        test_scaled_df  = pd.DataFrame(test_scaled,  columns=feature_columns, index=test.index )

        # Concatenate the scaled features with the excluded columns
        train = pd.concat([train_scaled_df, train[exclude_cols]], axis=1)
        val   = pd.concat([val_scaled_df,   val[exclude_cols]],   axis=1)
        test  = pd.concat([test_scaled_df,  test[exclude_cols]],  axis=1)

        X_train, y_train = reshape_data_rnn(train, feature_columns, max_slices=self.max_slices, sortby_column='slice_index', target_column='target')
        X_val,   y_val   = reshape_data_rnn(val,   feature_columns, max_slices=self.max_slices, sortby_column='slice_index', target_column='target')
        X_test,  y_test  = reshape_data_rnn(test,  feature_columns, max_slices=self.max_slices, sortby_column='slice_index', target_column='target')

        sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

        model = self.build_model((X_train.shape[1], X_train.shape[2]))

        # Train the model
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=self.patience, restore_best_weights=True)],
            verbose=0,  # Reduced verbosity for parallel execution
            sample_weight=sample_weight
        )

        y_train_pred, train_report = report(model=model, x=X_train, y_true=y_train, unique_classes=[2, 6.5, 8.5])
        y_val_pred,   val_report   = report(model=model, x=X_val,   y_true=y_val,   unique_classes=[2, 6.5, 8.5])
        y_test_pred,  test_report  = report(model=model, x=X_test,  y_true=y_test,  unique_classes=[2, 6.5, 8.5])

        # Store results to return
        local_results = {
            'test': test_subject_idx,
            'val': validation_subjects_idx,
            'history': history.history,
            'train_report': train_report,
            'y_train': y_train,
            'y_train_pred': y_train_pred,
            'val_report': val_report,
            'y_val': y_val,
            'y_val_pred': y_val_pred,
            'test_report': test_report,
            'y_test': y_test,
            'y_test_pred': y_test_pred,
        }
        
        logging.info(f"epoch : {len(history.history['val_loss'])}")
        logging.info(f"Completed Test {test_subject_idx} & Validation {validation_subjects_idx}")
        logging.info('=======================================')

        return local_results
            

    def test_model(self, data):
        unique_subject_id = data['subject_id'].unique()

        logging.info(f"Found {len(unique_subject_id)} valid test-validation pairs.")

        # Prepare arguments for parallel processing
        args_list = [(self, i, data) for i in unique_subject_id]
        random.shuffle(args_list)

        output_folder_name = datetime.now().strftime("%Y-%m-%d|%H:%M:%S|RNN")
        output_parent_dir = "/home/mojtaba.kananisarche/Zh/UTA/Output/"
        output_folder_path = os.path.join(output_parent_dir, output_folder_name)
        os.makedirs(output_folder_path, exist_ok=True)

        with open(os.path.join(output_folder_path, 'results.json'), 'w') as file:
            json.dump({
                'general':{
                    'units':self.units,
                    'dropout_ratio':self.dropout_ratio,
                    'batch_size':self.batch_size,
                    'learning_rate':self.learning_rate,
                    'weight_decay':self.weight_decay,
                    'epochs':self.epochs,
                    'dataset_id':self.dataset_id,
                    'max_slices':self.max_slices
                }
            }, file)

        # Run training in parallel and process results as they come
        all_results = []

        # with concurrent.futures.ProcessPoolExecutor(max_workers=1) as executor:
        #     futures = [executor.submit(self._train_single_fold, args) for args in args_list]
        #     for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Training folds"):
        #         result = future.result()

        results = [self._train_single_fold(args) for args in args_list]
        for result in results:
            file_path = os.path.join(output_folder_path, f'{result["test"]}.json')
            with open(file_path, 'w') as file:
                json.dump(result, file, cls=NumpyEncoder, indent=4)
                logging.info(f"Saved results for Test subject {result['test']} to {file_path}")
                
                    
                # Process results
        for result in all_results:
            self.models.append(result['model'])
            self.histories.append(result['history'])
            self.reports.append({
                'test': result['test'],
                'val': result['val'],
                'train_report': result['train_report'],
                'val_report': result['val_report'],
                'test_report': result['test_report']
            })
        logging.info(f"Completed all {len(unique_subject_id)} test-validation pairs.")



In [ ]:
m = Model(units = [1], 
        dropout_ratio = 0.3, 
        kernel_regularizer_l2 = 0, 
        kernel_regularizer_l1 = 0, 
        batch_size = 32, 
        num_classes = 3, 
        learning_rate = 1e-3, 
        weight_decay = 1e-4, 
        beta_1 = 0.9, 
        beta_2 = 0.999, 
        epsilon = 1e-7,
        epochs = 1, 
        patience = 100,
        max_slices=50,
        dataset_name = dataset_name)

m.test_model(df)

2025-03-10 01:27:06,136 - INFO - Found 60 valid test-validation pairs.
2025-03-10 01:27:06,173 - INFO - Starting Test 42 & Validation [39, 38, 37, 40, 43, 44, 45, 46, 47, 41, 48]
2025-03-10 01:27:06,198 - INFO - train (27222, 285) & val (5101, 285) & test (360, 285)
2025-03-10 01:27:10,450 - INFO - epoch : 1
2025-03-10 01:27:10,451 - INFO - Completed Test 42 & Validation [39, 38, 37, 40, 43, 44, 45, 46, 47, 41, 48]
2025-03-10 01:27:10,452 - INFO - =======================================
2025-03-10 01:27:10,457 - INFO - Saved results for Test subject 42 to /home/mojtaba.kananisarche/Zh/UTA/Output/2025-03-10|01:27:06|RNN/42.json
2025-03-10 01:27:10,459 - INFO - Completed all 1 test-validation pairs.
